# 4. Suppression baselines and the side channel

`system_prompt`, `refusal_classifier` and `lora_ga` all suppress. Watch what suppression
does to detectability.

In [ ]:
# On Kaggle or Colab, uncomment to install
# !pip install -q -e /kaggle/working/silentwall
# !pip install -q -e .

from silentwall.config import load_config
from silentwall.pipeline import prepare_workspace, run_method, run_sweep, save_workspace
from silentwall.report.render import render_comparison, render_markdown, write_comparison

CONFIG = "../configs/smoke.yaml"
cfg = load_config(CONFIG)
print(cfg.profile, cfg.tier, "methods:", len(cfg.methods))

In [ ]:
ws = prepare_workspace(cfg, verbose=False)

results = []
for method_id in ("clean_reference", "system_prompt", "refusal_classifier", "lora_ga"):
    results.append(run_method(ws, method_id))

In [ ]:
print(render_comparison(results))

## The point

`refusal_classifier` leaks nothing. It is also the most detectable method in the set.

An observer who probes the agent with ordinary questions can see which companies it has
gone quiet on, and reconstruct the restricted list without ever requesting a single
protected fact. In a compliance setting that list is the confidential thing.

Suppression is what creates the signature, so suppressing harder makes this worse rather
than better.

In [ ]:
for r in results:
    det = r.primary_detectability
    if det is None:
        continue
    print(f"--- {r.method_id} ---")
    ranked = sorted(det.feature_importance.items(), key=lambda kv: -abs(kv[1].point))
    for name, iv in ranked[:4]:
        print(f"  {name:24s} {iv}")
    print()